# Rapport d'Évaluation - Modèle Baseline
## Classification de Matériaux Thermiques - Sprint 1

Ce notebook présente une analyse complète des résultats d'entraînement et d'évaluation du modèle baseline pour la classification de matériaux thermiques.

**Date**: Décembre 2024  
**Modèle**: Baseline CNN  
**Dataset**: Thermal Waste Detection (Roboflow) - 707 images, 7 classes


In [ ]:
# Imports
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Chemins
PROJECT_ROOT = Path('..')
MODEL_DIR = PROJECT_ROOT / 'models' / 'baseline'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
DATA_DIR = PROJECT_ROOT / 'data'

print("Imports réussis")


## 1. Chargement des données d'évaluation


In [ ]:
# Charger les métriques d'évaluation
with open(OUTPUTS_DIR / 'baseline_model_metrics.json', 'r') as f:
    metrics = json.load(f)

# Charger l'historique d'entraînement
with open(MODEL_DIR / 'baseline_model_history.json', 'r') as f:
    history = json.load(f)

# Charger le résumé du modèle
with open(MODEL_DIR / 'baseline_model_summary.json', 'r') as f:
    model_summary = json.load(f)

# Charger le mapping des classes
with open(MODEL_DIR / 'baseline_model_class_mapping.json', 'r') as f:
    class_mapping = json.load(f)

# Inverser le mapping pour avoir classe -> index
index_to_class = {v: k for k, v in class_mapping.items()}
class_names = [index_to_class[i] for i in range(len(class_mapping))]

print("Données chargées avec succès")
print(f"\nMétriques globales:")
print(f"  - Accuracy: {metrics['accuracy']:.2%}")
print(f"  - Precision (macro): {metrics['precision_macro']:.2%}")
print(f"  - Recall (macro): {metrics['recall_macro']:.2%}")
print(f"  - F1-score (macro): {metrics['f1_macro']:.2%}")


## 2. Courbes d'entraînement


In [ ]:
# Extraire les données d'historique
epochs = history['epoch']
train_loss = history['loss']
train_acc = history['accuracy']
val_loss = history['val_loss']
val_acc = history['val_accuracy']

# Créer les figures
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Courbes d\'entraînement - Modèle Baseline', fontsize=16, fontweight='bold')

# Loss
axes[0, 0].plot(epochs, train_loss, 'b-', label='Train Loss', linewidth=2)
axes[0, 0].plot(epochs, val_loss, 'r-', label='Validation Loss', linewidth=2)
axes[0, 0].axvline(x=model_summary['metrics']['best_epoch'], 
                   color='g', linestyle='--', alpha=0.7, label=f"Meilleur epoch ({model_summary['metrics']['best_epoch']})")
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Évolution de la Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(epochs, train_acc, 'b-', label='Train Accuracy', linewidth=2)
axes[0, 1].plot(epochs, val_acc, 'r-', label='Validation Accuracy', linewidth=2)
axes[0, 1].axvline(x=model_summary['metrics']['best_epoch'], 
                   color='g', linestyle='--', alpha=0.7, label=f"Meilleur epoch ({model_summary['metrics']['best_epoch']})")
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Évolution de l\'Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision
if 'precision' in history and 'val_precision' in history:
    axes[1, 0].plot(epochs, history['precision'], 'b-', label='Train Precision', linewidth=2)
    axes[1, 0].plot(epochs, history['val_precision'], 'r-', label='Validation Precision', linewidth=2)
    axes[1, 0].axvline(x=model_summary['metrics']['best_epoch'], 
                       color='g', linestyle='--', alpha=0.7)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Évolution de la Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

# Recall
if 'recall' in history and 'val_recall' in history:
    axes[1, 1].plot(epochs, history['recall'], 'b-', label='Train Recall', linewidth=2)
    axes[1, 1].plot(epochs, history['val_recall'], 'r-', label='Validation Recall', linewidth=2)
    axes[1, 1].axvline(x=model_summary['metrics']['best_epoch'], 
                       color='g', linestyle='--', alpha=0.7)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Évolution du Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'baseline_model_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Courbes sauvegardées: {OUTPUTS_DIR / 'baseline_model_training_curves.png'}")


In [ ]:
# Créer un DataFrame pour les métriques globales
global_metrics = {
    'Métrique': ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1-score (macro)'],
    'Valeur': [
        metrics['accuracy'],
        metrics['precision_macro'],
        metrics['recall_macro'],
        metrics['f1_macro']
    ]
}
df_global = pd.DataFrame(global_metrics)

# Visualisation
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(df_global['Métrique'], df_global['Valeur'], 
               color=['#2ecc71', '#3498db', '#9b59b6', '#e74c3c'])
ax.set_xlabel('Score', fontsize=12)
ax.set_title('Métriques Globales d\'Évaluation', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1)
ax.grid(True, alpha=0.3, axis='x')

# Ajouter les valeurs sur les barres
for i, (bar, val) in enumerate(zip(bars, df_global['Valeur'])):
    ax.text(val + 0.01, i, f'{val:.2%}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'baseline_model_global_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print(df_global.to_string(index=False))


## 4. Performance par classe


In [ ]:
# Créer un DataFrame pour les métriques par classe
per_class_data = []
for class_name in class_names:
    class_metrics = metrics['per_class'][class_name]
    per_class_data.append({
        'Classe': class_name,
        'Precision': class_metrics['precision'],
        'Recall': class_metrics['recall'],
        'F1-score': class_metrics['f1_score']
    })

df_per_class = pd.DataFrame(per_class_data)

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Performance par Classe', fontsize=16, fontweight='bold')

metrics_to_plot = ['Precision', 'Recall', 'F1-score']
colors = ['#3498db', '#2ecc71', '#e74c3c']

for idx, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    ax = axes[idx]
    bars = ax.barh(df_per_class['Classe'], df_per_class[metric], color=color)
    ax.set_xlabel(metric, fontsize=12)
    ax.set_title(f'{metric} par Classe', fontsize=13)
    ax.set_xlim(0, 1)
    ax.grid(True, alpha=0.3, axis='x')
    
    # Ajouter les valeurs
    for i, (bar, val) in enumerate(zip(bars, df_per_class[metric])):
        ax.text(val + 0.01, i, f'{val:.2%}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'baseline_model_per_class_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nMétriques par classe:")
print(df_per_class.to_string(index=False))


## 5. Matrice de confusion


In [ ]:
# Charger la matrice de confusion
cm = np.array(metrics['confusion_matrix'])

# Créer la visualisation
fig, ax = plt.subplots(figsize=(10, 8))

# Normaliser la matrice pour les pourcentages
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Créer l'heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, cbar_kws={'label': 'Nombre de prédictions'})

ax.set_xlabel('Prédictions', fontsize=12, fontweight='bold')
ax.set_ylabel('Vraies classes', fontsize=12, fontweight='bold')
ax.set_title('Matrice de Confusion - Modèle Baseline', fontsize=14, fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'baseline_model_confusion_matrix_detailed.png', dpi=300, bbox_inches='tight')
plt.show()

# Afficher aussi la version normalisée
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, cbar_kws={'label': 'Pourcentage'})
ax.set_xlabel('Prédictions', fontsize=12, fontweight='bold')
ax.set_ylabel('Vraies classes', fontsize=12, fontweight='bold')
ax.set_title('Matrice de Confusion Normalisée (Pourcentages)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'baseline_model_confusion_matrix_normalized.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nStatistiques de la matrice de confusion:")
print(f"  - Total de prédictions: {cm.sum()}")
print(f"  - Prédictions correctes: {np.trace(cm)}")
print(f"  - Prédictions incorrectes: {cm.sum() - np.trace(cm)}")


## 6. Analyse des erreurs


In [ ]:
# Analyser les erreurs les plus fréquentes
error_analysis = []
for i, true_class in enumerate(class_names):
    for j, pred_class in enumerate(class_names):
        if i != j and cm[i, j] > 0:
            error_analysis.append({
                'Vraie classe': true_class,
                'Prédiction': pred_class,
                'Nombre': cm[i, j],
                'Pourcentage': (cm[i, j] / cm[i, :].sum()) * 100
            })

df_errors = pd.DataFrame(error_analysis).sort_values('Nombre', ascending=False)

print("Top 10 des erreurs les plus fréquentes:")
print(df_errors.head(10).to_string(index=False))

# Visualisation des erreurs
if len(df_errors) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))
    top_errors = df_errors.head(15)
    error_labels = [f"{row['Vraie classe']} → {row['Prédiction']}" 
                    for _, row in top_errors.iterrows()]
    bars = ax.barh(range(len(top_errors)), top_errors['Nombre'], color='#e74c3c')
    ax.set_yticks(range(len(top_errors)))
    ax.set_yticklabels(error_labels)
    ax.set_xlabel('Nombre d\'erreurs', fontsize=12)
    ax.set_title('Top 15 des Erreurs de Classification', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Ajouter les valeurs
    for i, (bar, val) in enumerate(zip(bars, top_errors['Nombre'])):
        ax.text(val + 0.1, i, f'{int(val)}', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'baseline_model_error_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()


## 7. Résumé et conclusions

### Points forts
- Modèle baseline fonctionnel et reproductible
- Pipeline DVC complet et validé
- Classes `glass` et `organic` : Performance modérée (F1 ≈ 0.47-0.48)
- Classe `plastic` : Bonne précision (0.67)

### Points à améliorer
- Performance globale faible (accuracy 39.4%)
- Classes `aluminium` et `paper` : Aucune prédiction correcte
- Signes de surapprentissage (écart train/validation)
- Déséquilibre entre précision et rappel

### Recommandations pour Sprint 2
1. **Augmentation de données** : Activer l'augmentation pour réduire le surapprentissage
2. **Architecture** : Modèle plus profond (ResNet, EfficientNet) ou transfer learning
3. **Gestion du déséquilibre** : Weighted loss, oversampling, focal loss
4. **Hyperparamètres** : Optimisation du learning rate et régularisation
5. **Dataset** : Collecte de plus de données pour classes rares (aluminium, paper)

### Objectifs Sprint 2
- Accuracy > 60%
- F1-score macro > 0.55
- Amélioration des classes sous-représentées
